# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, overview, analysis, and visualization of the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all data entities by their `@id` per Croissant/FAIR best practice.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display basic metadata (name, description, version)
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}\n")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Enumerate and inspect the high-level data structure with Croissant entity identifiers. If the schema contains more than one record set, show all; otherwise, show the primary record set.

In [ ]:
# List all record sets, fields, and example columns with their @id
from collections import defaultdict

print("Available Record Sets (by @id):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- {rs['@id']} (name: {rs.get('name', '[no name]')})")

# Inspect the first record set in detail
if record_sets:
    primary_record_set_id = record_sets[0]['@id']
    print(f"\nExploring primary Record Set: {primary_record_set_id}")
    # List fields in the record set
    print("  Fields:")
    for field in record_sets[0].get('field', []):
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)} (name: {field.get('name', '[no name]')})")
        else:
            print(f"    - {field}")
else:
    print("No record sets found in this schema.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s as reviewed above for indexing and reference.

In [ ]:
# Extract data for all available record sets by @id
dataframes = dict()
selected_record_set_id = None

for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
    else:
        df = pd.DataFrame()
    dataframes[rs_id] = df
    if selected_record_set_id is None and not df.empty:
        selected_record_set_id = rs_id

if selected_record_set_id:
    print(f"Loaded records from Record Set: {selected_record_set_id}\nColumns:")
    print(dataframes[selected_record_set_id].columns.to_list())
    display(dataframes[selected_record_set_id].head())
else:
    print("No data extracted from any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply exploratory steps to the chosen record set.

We'll select a representative numeric field (for example, one ending with `Count`, `Abundance`, or `MW [Da]`) and filter and normalize its values. All field references will be by `@id` (column name in the DataFrame).

In [ ]:
# EDA using a numeric field

import numpy as np

# Manually choose a suitable numeric field based on name heuristics for demonstration
df = dataframes[selected_record_set_id]
numeric_candidates = [c for c in df.columns if any(k in c.lower() for k in ['count', 'number', 'abundance', 'mw', 'coverage', 'quantity', 'score', 'peptide'])]

numeric_field_id = numeric_candidates[0] if numeric_candidates else df.select_dtypes(np.number).columns[0]
print(f"Numeric field selected (by @id): {numeric_field_id}")

# Filter: values > threshold
threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered rows where {numeric_field_id} > {threshold}")
display(filtered_df[[numeric_field_id]].head())

# Normalization
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} (z-score):")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optional grouping: use a likely categorical field (e.g. containing 'description', 'class', 'type')
group_candidates = [c for c in df.columns if any(k in c.lower() for k in ['desc', 'group', 'type', 'condition', 'category', 'accession', 'sample'])]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and/or its relationship with the grouping field (if present).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the raw numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot grouped by group_field if it exists
if 'group_field' in locals():
    plt.figure(figsize=(12, 6))
    order = df[group_field].value_counts().index[:10]  # Show top 10 groups
    sns.boxplot(data=df, x=group_field, y=numeric_field_id, order=order)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=60)
    plt.show()


## 6. Conclusion
In this notebook, we've:

- Loaded and described the FAIR^2 Croissant dataset package for mass spectrometry analysis of extracellular vesicles from human mast cells.
- Explored the schema, inspecting record sets, fields, and their stable `@id`s for reproducibility.
- Extracted data for analysis into DataFrames indexed by Croissant `@id`.
- Performed basic EDA by filtering, normalizing, and grouping on representative fields.
- Visualized distributions and (if available) relationships between fields.

Refer to the Croissant metadata and identifiers for precise programmatic and semantic data access in downstream applications.